## Spatial changes in Asterix over time? 
------------------------------------------
To test for spatial changes i Asterix between first scan and 13th scan of the day 
to do it, i'm averaging sub 15> images and then 
### Nilearn Smoothing 
https://nilearn.github.io/dev/modules/generated/nilearn.image.smooth_img.html 

Apply a Gaussian filter along the three first dimensions of arr. In all cases, non-finite values in input image are replaced by zeros.

Parameters:

    imgsNiimg-like object, SurfaceImage, or iterable of Niimg-like objects or SurfaceImage.

        Image(s) to smooth (see Input and output: neuroimaging data representation for a detailed description of the valid input types).
    fwhmscalar, numpy.ndarray, or tuple, or list,or ‘fast’ or None, optional

        Smoothing strength, as a full-width at half maximum, in millimeters.

        For surface data, only scalar and None are supported.

        For volume data, several options are possible:

            If a nonzero scalar is given, width is identical in all 3 directions.

            If a numpy.ndarray, tuple, or list is given, it must have 3 elements, giving the FWHM along each axis. If any of the elements is 0 or None, smoothing is not performed along that axis.

            If fwhm=”fast”, a fast smoothing will be performed with a filter [0.2, 1, 0.2] in each direction and a normalization to preserve the local average value.

            If fwhm is None, no filtering is performed (useful when just removal of non-finite values is needed).


In [ ]:
import os
import glob
import subprocess


def get_sessions(prj_path, subj_id):
    """Return sorted list of session folder names found for a subject in derivatives/"""
    subj_dir = os.path.join(prj_path, "derivatives", subj_id)
    ses_dirs = sorted(glob.glob(os.path.join(subj_dir, "ses-*")))
    return [os.path.basename(d) for d in ses_dirs]

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
prj_path=r"/home/malberti/wks14/temp/FF_DWI_Drift" # main BIDS path /home/malberti/Unix_Folders/Sweep/Pilot/SW005
subject_ids = ["sub-44", "sub-45"] #, "sub-36", "sub-37", "sub-38", "sub-39", "sub-40", "sub-9998", "sub-9999"]  # ← fill in your list "sub-33", "sub-34", "sub-35", "sub-36", "sub-37", 

PANORAMIX   = "/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean.nii"
mask        = "/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean_mask.nii"
METRICS      = ["FA", "MD"]
l2fwhm      = [1.69865806, 2.54798709]   # ≈ 4 mm / 6 mm FWHM in MNI space
kernel_sizes = ["4mm", "6mm"]

# ── Run ───────────────────────────────────────────────────────────────────────

for metric in METRICS: 
    for pid in subject_ids:
        sessions = get_sessions(prj_path, pid)
        print(f"{pid}: found {len(sessions)} sessions -> {sessions}")

        for ses in sessions:
            subjid = f"{pid}_{ses}"
            base       = os.path.join(prj_path, "derivatives", pid, ses, "dwi")
            fa_path    = os.path.join(base, "index", "Tensor", f"{subjid}_prep_FA.nii")
            map_path   = os.path.join(base, "index", "Tensor", f"{subjid}_prep_{metric}.nii")

            coreg_dir  = os.path.join(base, "index2TEMPLATE")
            smooth_dir = os.path.join(coreg_dir, "smooth")
            os.makedirs(smooth_dir, exist_ok=True)

            mat_out    = os.path.join(coreg_dir, f"{subjid}_prep_FA2PANORAMIX.mat")
            coreg_out  = os.path.join(coreg_dir, f"{subjid}_prep_{metric}2PANORAMIX.nii")
            smooth_out = os.path.join(smooth_dir, f"{subjid}_prep_{metric}2PANORAMIX-{kernel_sizes[0]}.nii")
            tmp1       = os.path.join(smooth_dir, "tmp1.nii")
            tmp2       = os.path.join(smooth_dir, "tmp2.nii")

            if not os.path.exists(fa_path):
                print(f"  ! Missing FA map for {subjid}, skipping")
                print(fa_path)
                continue

            print(f"Coreg: {subjid}")
          #  subprocess.run(f"flirt -in {fa_path} -ref {PANORAMIX} -dof 6 -omat {mat_out}", shell=True, check=True)
          #  subprocess.run(f"flirt -in {map_path} -ref {PANORAMIX} -applyxfm -init {mat_out} -out {coreg_out}", shell=True, check=True)

          # Estimate rigid (6 DOF) transform: FA → PANORAMIX template
            out_prefix = os.path.join(coreg_dir, f"{subjid}_")
            subprocess.run(
                f"antsRegistrationSyNQuick.sh -d 3 -t r -f {PANORAMIX} -m {fa_path} -o {out_prefix}",
                shell=True, check=True
            )

            mat_out = f"{out_prefix}0GenericAffine.mat"

            # Apply the same transform to the metric map
            subprocess.run(
                f"antsApplyTransforms -d 3 -i {map_path} -r {PANORAMIX} -o {coreg_out} -t {mat_out} -n Linear",
                shell=True, check=True
            )

In [4]:
# ── Config ───────────────────────────────────────────────────────────────────
prj_path=r"/home/malberti/wks14/temp/FF_DWI_Drift" # main BIDS path /home/malberti/Unix_Folders/Sweep/Pilot/SW005
#subject_ids = ["sub-41", "sub-42", "sub-43", "sub-44"] #, "sub-36", "sub-37", "sub-38", "sub-39", "sub-40", "sub-9998", "sub-9999"]  # ← fill in your list "sub-33", "sub-34", "sub-35", "sub-36", "sub-37", 

PANORAMIX   = "/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean.nii"
mask        = "/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean_mask.nii"
METRICS      = ["FA", "MD"]
l2fwhm      = [1.69865806, 2.54798709]   # ≈ 4 mm / 6 mm FWHM in MNI space
kernel_sizes = ["4mm", "6mm"]

# ── Run ───────────────────────────────────────────────────────────────────────

for metric in METRICS: 
    for pid in subject_ids:
        sessions = get_sessions(prj_path, pid)
        print(f"{pid}: found {len(sessions)} sessions -> {sessions}")

        for ses in sessions:
            subjid = f"{pid}_{ses}"
            base       = os.path.join(prj_path, "derivatives", pid, ses, "dwi")
            fa_path    = os.path.join(base, "index", "Tensor", f"{subjid}_prep_FA.nii")
            map_path   = os.path.join(base, "index", "Tensor", f"{subjid}_prep_{metric}.nii")

            coreg_dir  = os.path.join(base, "index2TEMPLATE")
            smooth_dir = os.path.join(coreg_dir, "smooth")
            os.makedirs(smooth_dir, exist_ok=True)

            mat_out    = os.path.join(coreg_dir, f"{subjid}_prep_FA2PANORAMIX.mat")
            coreg_out  = os.path.join(coreg_dir, f"{subjid}_prep_{metric}2PANORAMIX.nii")
            smooth_out = os.path.join(smooth_dir, f"{subjid}_prep_{metric}2PANORAMIX-{kernel_sizes[1]}.nii")
            tmp1       = os.path.join(smooth_dir, "tmp1.nii")
            tmp2       = os.path.join(smooth_dir, "tmp2.nii")

            if not os.path.exists(fa_path):
                print(f"  ! Missing FA map for {subjid}, skipping")
                print(fa_path)
                continue

            print(f"Coreg: {subjid}")
          #  subprocess.run(f"flirt -in {fa_path} -ref {PANORAMIX} -dof 6 -omat {mat_out}", shell=True, check=True)
          #  subprocess.run(f"flirt -in {map_path} -ref {PANORAMIX} -applyxfm -init {mat_out} -out {coreg_out}", shell=True, check=True)

          #   
            subprocess.run(f"fslmaths {coreg_out} -mas {mask} -s {l2fwhm[1]} -mas {mask} {tmp1}", shell=True, check=True)
            subprocess.run(f"fslmaths {mask} -s {l2fwhm[1]} -mas {mask} {tmp2}", shell=True, check=True)
            subprocess.run(f"fslmaths {tmp1} -div {tmp2} {smooth_out}", shell=True, check=True)
         #   subprocess.run(f"fsleyes {coreg_out} {smooth_out} {mask}", shell=True, check=True)
            subprocess.run(f"rm -f {tmp1} {tmp2}", shell=True, check=True)

sub-44: found 4 sessions -> ['ses-01', 'ses-02', 'ses-03', 'ses-04']
Coreg: sub-44_ses-01
Coreg: sub-44_ses-02
Coreg: sub-44_ses-03
Coreg: sub-44_ses-04
sub-45: found 13 sessions -> ['ses-01', 'ses-02', 'ses-03', 'ses-04', 'ses-05', 'ses-06', 'ses-07', 'ses-08', 'ses-09', 'ses-10', 'ses-11', 'ses-12', 'ses-13']
Coreg: sub-45_ses-01
Coreg: sub-45_ses-02
Coreg: sub-45_ses-03
Coreg: sub-45_ses-04
Coreg: sub-45_ses-05
Coreg: sub-45_ses-06
Coreg: sub-45_ses-07
Coreg: sub-45_ses-08
Coreg: sub-45_ses-09
Coreg: sub-45_ses-10
Coreg: sub-45_ses-11
Coreg: sub-45_ses-12
Coreg: sub-45_ses-13
sub-44: found 4 sessions -> ['ses-01', 'ses-02', 'ses-03', 'ses-04']
Coreg: sub-44_ses-01
Coreg: sub-44_ses-02
Coreg: sub-44_ses-03
Coreg: sub-44_ses-04
sub-45: found 13 sessions -> ['ses-01', 'ses-02', 'ses-03', 'ses-04', 'ses-05', 'ses-06', 'ses-07', 'ses-08', 'ses-09', 'ses-10', 'ses-11', 'ses-12', 'ses-13']
Coreg: sub-45_ses-01
Coreg: sub-45_ses-02
Coreg: sub-45_ses-03
Coreg: sub-45_ses-04
Coreg: sub-45_ses